In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import SupabaseVectorStore
from supabase.client import Client, create_client

supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_PRIVATE_KEY")
supabase: Client = create_client(supabase_url, supabase_key)

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004",google_api_key=os.getenv("GOOGLE_API_KEY"), )

/Users/alexandersiladie/Desktop/ScribeLec/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain.schema import Document

loader = TextLoader("transcript2.txt")
documents = loader.load()
pattern = r"\[(\d+\.\d+s)\]:\s*(.*)"


In [6]:
import re
docs = []
for doc in documents:
    for line in doc.page_content.splitlines():
        match = re.match(pattern, line)
        if match:
            timestamp = match.group(1)  # Extract the timestamp
            timestamp = timestamp[:-1]
            text = match.group(2)  # Extract the text
            if text.strip():  # Only process non-empty lines
                # Create a document with the text and metadata (timestamp)
                docs.append(Document(page_content=text, metadata = {"timestamp": timestamp, "lecture": "3ae57d24-8426-44c3-816b-f23e3ae04d0b"}
                ))


In [4]:
vector_store = SupabaseVectorStore.from_documents(
    docs,
    embeddings,
    client=supabase,
    table_name="documents",
    query_name="match_documents"
)